# CPSC 4770/5770 Spring 2026

Instructor: Arman Cohan  
Homework 2: Transformers and language modeling

### Part 1: Implementation of a transformer model for language modeling
(60 points total)

### Please write your name and NetID below

NAME:

NetID:



**IMPORTANT SUBMISSION NOTE**: Please submit your notebook having run all cells and hide long debugging output before submission. If you included print or debugging statements, please remove those before your final run and submission.

**AI policy**: Do not use AI tools (e.g., Anthropic CC, OpenAI Codex, Cursor, ChatGPT, Copilot, Gemini, etc) to generate, complete, or debug your code.
Basic IDE autocomplete that finishes syntax (e.g., closing brackets, completing a for/while/if-else template) is fine, but AI-driven code suggestions that implement logic or functionality are not permitted.
If using Colab, please turn off "Show AI-powered inline completions" in your settings.

---

## Introduction

In this assingment we will implement the transformer model architecture for language modeling from scratch.  

## Environment Setup

For this assignment, we will use Google Colab.

#### Using GPU in Colab
PyTorch and other deep learning libraries are much faster using GPU acceleration. For training and evaluating the models in this assignment, you should always use a GPU:

1. Go to __Runtime__ option on the top left
2. Click __Change runtime type__
3. Select "GPU" for __Hardware
 accelerator__
4. Click __SAVE__ button

However, Colab limits the amount of time that you can use a free GPU.
So you may wish to implement much of the assignment without the GPU. But note that you will have to run all cells again once you change the runtime type.
You can also connect Colab to your local GPU for faster iteration.

Alternatively the course is already setup on HPC, so you can access the jupyter notebook functionality there and run your code there.

Colab has popular libraries already installed such as Pytorch, TensorFlow, OpenCV and Keras. Let's get started and verify this:

In [ ]:
! pip install transformers tokenizers
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
from typing import Tuple, List

import random
import math
import os
import time
import json
import numpy as np

# We'll set the random seeds for deterministic results.
SEED = 1

random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.enabled = False
torch.backends.cudnn.deterministic = True

class Placeholder:
    @property
    def DO(self):
        raise NotImplementedError("You haven't yet implemented this part of the assignment yet")

TO = Placeholder()

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Pytorch version is: ", torch.__version__)
print("You are using: ", DEVICE)

# Transformers

First, we will take a look into the famous *Transformer* model, which is foundation of LLMs such as current GPT-4o, Claude and more conventional models such as GPT-3, BERT, etc.
We will use it in the following parts of this assignment.

Transformers were introduced in the paper ["Attention is all you need" (Vaswani et al. 2017)](https://arxiv.org/abs/1706.03762). As the paper title suggests, the key idea that makes transformers work is *attention*. If you want to review attention and transformers, some useful resources include

- the [original paper](https://arxiv.org/abs/1706.03762)
- chapters 9 and 10 of Jurafsky & Martin
- the blog posts [Visualizing A Neural Machine Translation Model](https://jalammar.github.io/visualizing-neural-machine-translation-mechanics-of-seq2seq-models-with-attention/) and [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/). (The first of these doesn't cover transformers but is useful for understanding attention.)
http://nlp.seas.harvard.edu/annotated-transformer/
- Youtube videos such as [this one](https://youtu.be/OyFJWRnt_AY) or [this one](https://youtu.be/iDulhoQ2pro).

## 1 Implementing the model input pipeline

Recall that in transfomer models, the input is a sequence of tokens.
Concretely here is the input pipeline for a transformer model (or most of the neural network models in NLP):

1- Given an input text $x$, we first tokenize it into a sequence of tokens. Tokens can be words or sub-word units (or even characters).
We can assume that we have access to a blackbox tokenization method that given an input text $x$, returns a sequence of tokens $x_1, x_2, \ldots, x_n$.

2- The tokens are then converted into a sequence of token IDs. Each token ID is an integer that represents the token in the vocabulary.

3- The token IDs are then converted into a sequence of token embeddings. Each token is represented as a vector, and the sequence of vectors is called an *embedding*.

4- We can also optionally include additional information such as the position of each token in the sequence. This is done by adding a position embedding to the token embedding.
Here each position is an integer that represents the position of the token in the sequence and then a separate position embedding matrix is used to look up the position embedding for each token.

Each token is represented as a vector, and the sequence of vectors is called an *embedding*.

### 1.1 Tokenization

We assume we have access to a blackbox tokenization method that given an input text $x$, returns a sequence of tokens $x_1, x_2, \ldots, x_n$.
For this we use the `tokenizers` library from HuggingFace. This library provides a wide range of tokenizers for different languages and models.
We will use the `GPT2Tokenizer` for this assignment.

In [ ]:
from transformers import GPT2Tokenizer

# Load pre-trained model tokenizer (vocabulary)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Encode a text inputs
text = "This adaptation of the enigmatic novel by Liane Moriarty is supremely watchable but flawed."

# we can use tokenizer.encode to convert text to token IDs
tokens_ids = tokenizer.encode(text)

tokens_ids


In [ ]:
# now convert token IDs back to text to inspect the tokens
tokens = tokenizer.convert_ids_to_tokens(tokens_ids)

tokens


You will notice some tokens start with a "Ġ".
Popular tokenizers use a special symbol such as "Ġ" (BPE tokenizer such as GPT2) or "▁" (SentencePiece) to represent space.

#### Converting token IDs to text

We can also use the tokenizers library to convert token IDs back to text. This is useful for represting the output of the model in human readable form.

In [ ]:
# we can use tokenizer.decode() method to convert token IDs back to text

converted_text = tokenizer.decode(tokens_ids)
print(converted_text)

assert text == converted_text

### 1.2 Embeddings [3 pts]

**Rubric:** Implement `Embedding` class with `nn.Embedding` and forward pass.

We first need to implement the embedding layer of the transformer. Recall that this is layer 0. We will use the `nn.Embedding` layer in PyTorch to implement this.
The `nn.Embedding` creates an embdding matrix of size `vocab_size x embedding_dim`. Given a sequence of token IDs, we can use the `nn.Embedding` layer to look up the token embeddings for given token IDs.

In [ ]:
import torch
import torch.nn as nn
from torch import Tensor

class Embedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        """
        Initializes the embedding layer.

        Args:
            vocab_size (int): The number of unique tokens in the vocabulary.
            d_model (int): The embedding dimension (size of each embedding vector).
        """
        super(Embedding, self).__init__()

        # 🔹 TODO: Define the embedding layer using nn.Embedding.
        # 💡 Hint: The nn.Embedding layer should map 'vocab_size' tokens into 'd_model' embeddings.

        self.wte = None  # 🔻 REPLACE 'None' with your implementation

    def forward(self, x: Tensor) -> Tensor:
        """
        Performs a forward pass of the embedding layer.

        Args:
            x (Tensor): A tensor of token indices with shape (batch_size, sequence_length).

        Returns:
            Tensor: The corresponding embeddings with shape (batch_size, sequence_length, d_model).
        """

        # 🔹 TODO: Lookup embeddings for the given input indices.
        # 💡 Hint: Use self.wte to retrieve embeddings.

        return None  # 🔻 REPLACE 'None' with your implementation


In [ ]:
# tests
vocab_size = 10
d_model = 16

embedding = Embedding(vocab_size, d_model)
x = torch.tensor([1, 2, 3, 4])
output = embedding(x)
assert output.shape == (4, d_model)


### 1.3 Positional Embeddings [2 pts]

**Rubric:** Implement positional embeddings using `nn.Embedding`.

Now we implement the positional embeddings. As mentioned above this is a learned position embedding method and we will use the `nn.Embedding` layer in PyTorch to implement this.

In [ ]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super(PositionalEmbeddings, self).__init__()

        # 🔹 TODO: Implement
        # create a tensor of shape (1, max_len, d_model) to store the positional embeddings using nn.Embedding
        self.positional_embeddings = None  # overwrite this line

    def forward(self, x: Tensor) -> Tensor:
        # add the positional embeddings to the input tensor
        # 🔹 TODO: Implement
        return None  # overwrite this line


In [ ]:
# tests

d_model = 16
max_len = 64

positional_embeddings = PositionalEmbeddings(d_model, max_len)
x = torch.tensor([1, 2, 3, 4])
output = positional_embeddings(x)
assert output.shape == (4, d_model)


Now we put the token embeddings and positional embeddings together to get the input embeddings for the transformer model.

In [ ]:
# Combine the embeddings and positional embeddings

class TokenEmbedder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, max_len: int):
        super(TokenEmbedder, self).__init__()
        self.token_embedding = Embedding(vocab_size, d_model)
        self.positional_embedding = PositionalEmbeddings(d_model, max_len)

    def forward(self, x: Tensor) -> Tensor:
        # add the token embeddings and positional embeddings together
        pos = torch.arange(0, x.shape[1], dtype=torch.long, device=x.device) # shape: [sequence length]
        return self.token_embedding(x) + self.positional_embedding(pos)

In [ ]:
# test the whole input pipeline

sample_texts = ["This adaptation of the enigmatic novel by Liane Moriarty is supremely watchable but flawed.",
                "The story is a bit of a slow burn, but the performances are top-notch and the ending is worth the wait."]

# encode the text
vocab_size = tokenizer.vocab_size
d_model = 64

# return_tensors="pt" returns pytorch tensors directly. truncation and padding are used to ensure the input length is the same
tokenizer.pad_token = tokenizer.eos_token
token_ids = tokenizer(sample_texts, return_tensors="pt", max_length=64, padding="longest", truncation=True)['input_ids']

token_embedder = TokenEmbedder(vocab_size=tokenizer.vocab_size, d_model=768, max_len=64)

# pass the token_ids to the token_embedder
output = token_embedder(token_ids)

output.shape

## 1.4 Reflection Questions [4 pts, 1pt each]

1- What is the purpose of the positional embeddings? Why do we need them?

🔹 TODO: Add your response here

2- What is the purpose of this line?   
`pos = torch.arange(0, x.shape[1], dtype=torch.long) # shape: [sequence length]`

🔹 TODO: Add your response here

3- What is the shape of the output from the TokenEmbedder correspond to?

🔹 TODO: Add your response here

4- Why do we add the positional embeddings to the token embeddings? Can we use other methods like concatenation?

🔹 TODO: Add your response here

In [ ]:
assert False, "Please answer the above questions and then comment this line to continue"

## 2. Transformer Model Architecture

Now that we have implemented the input pipeline we can move on to the transformer model architecture.

We will take a modular approach to implementing the transformer model. We will implement the following components of the transformer model:

1- QKV Projection

2- Multi-head self-attention

3- Position-wise feedforward network

4- Layer normalization

### 2.1 Projections [5 pts]

We first want to create separate projections for the input.
In class we implemented this:

In [ ]:
import torch
import torch.nn as nn

class QKVProjection(nn.Module):
    def __init__(self, d_model, num_heads, d_k):
        super(QKVProjection, self).__init__()
        assert num_heads * d_k == d_model, "d_model must be equal to num_heads * d_k"
        self.num_heads = num_heads
        self.d_k = d_k

        # 🔹 TODO: Create three nn.Linear layers (q_linear, k_linear, v_linear).
        # Each maps input dimension d_model -> output dimension (num_heads * d_k).
        self.q_linear = None  # hint: nn.Linear(?, ?)
        self.k_linear = None
        self.v_linear = None

    def forward(self, X):
        batch_size, seq_length, d_model = X.shape

        # --- How to go about implementing this---
        # After the linear layer, we have shape (batch_size, seq_length, num_heads * d_k).
        # we need to do some reshape and transpose and then do the projection **in a way that we can do batched attention**
        # For multi-head attention we need to run each head separately. So we:
        # 1. VIEW: split the last dim into (num_heads, d_k) -> (batch, seq, num_heads, d_k).
        # 2. TRANSPOSE(1, 2): swap seq and num_heads -> (batch, num_heads, seq, d_k).
        #    This way each "head" is (seq, d_k), and we can do batched attention:
        #    scores (batch, num_heads, seq, seq) @ V (batch, num_heads, seq, d_k) -> (batch, num_heads, seq, d_k).
        # Target shape for Q, K, V: (batch_size, num_heads, seq_length, d_k).

        # 🔹 TODO: Compute Q. Apply q_linear to X, then .view(..., num_heads, d_k), then .transpose(1, 2).
        Q = None  # self.q_linear(X).view(...).transpose(1, 2)

        # 🔹 TODO: Same for K and V (use k_linear and v_linear).
        K = None
        V = None

        return Q, K, V

# Example usage:
d_model = 512  # Embedding size
num_heads = 8  # Number of attention heads
d_k = 64  # Dimension per head (num_heads * d_k must be d_model)

batch_size = 2
seq_length = 10

# Random input tensor
X = torch.rand(batch_size, seq_length, d_model)

# Instantiate and apply QKVProjection
qkv_projection = QKVProjection(d_model, num_heads, d_k)
Q, K, V = qkv_projection(X)

print(Q.shape, K.shape, V.shape)

## 2.2 Multi-head self-attention [12 pts]

**Rubric:**
- Scaled dot-product attention scores [4 pts]
- Causal mask [2 pts]
- Padding mask [2 pts]
- Softmax, context, and output projection [4 pts]

The self-attention mechanism is the key idea that makes transformers work. It allows the model to weigh the importance of different tokens in the input sequence when computing the output for each token.
The main parameters of this layer are the projection matrices $W_Q, W_K, W_V$ and we have $H$ heads (for each head we have separate projections).
Recall that the dimension of the input sequence is $d_{model}$ and the dimension of the output sequence is also $d_{model}$.
And the dimension of the projected queries, keys and values is $d_k = d_v = d_{model} / H$.

Based on this information we can implement the multi-head self-attention layer.


**Note**: In tensor implementation, to help with readability of the code, we include the shape of the tensors in the tensor name.
For example if tensor `x` has shape `(batch_size, seq_len, d_model)` we will name it `x_bsd` where `b` is the batch size, `s` is the sequence length and `d` is the dimension of the tensor.

You should use this naming convention in your implementation.

Hint: For tensor multiplications you can use the `torch.einsum` operation that we discussed in the class.

In [ ]:
import math
import torch.nn.functional as F

# Large negative value for masking (more numerically stable than -inf, especially on MPS/Metal)
MASK_VALUE = -1e9

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, d_k, attn_pdrop=None):
        super(MultiHeadAttention, self).__init__()

        assert num_heads * d_k == d_model, "d_model must be equal to num_heads * d_k"

        self.num_heads = num_heads
        self.d_k = d_k
        self.attn_pdrop = attn_pdrop

        # Use the improved QKVProjection
        self.qkv_projection = QKVProjection(d_model, num_heads, d_k)

        # Final output projection
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, X, causal_mask=False, attention_mask=None):
        """
        Perform multi-head self-attention.

        Args:
            X (torch.Tensor): Input tensor of shape (batch_size, seq_length, d_model).
            causal_mask (bool): If True, apply a causal mask to prevent attending to future tokens.
            attention_mask (torch.Tensor, optional): Padding mask of shape (batch_size, seq_length).
                Contains 1 for real tokens and 0 for padding tokens.

        Returns:
            Tuple[torch.Tensor, torch.Tensor]: A tuple containing:
                - attention_weights (torch.Tensor): Attention weights of shape (batch_size, num_heads, seq_length, seq_length).
                - output (torch.Tensor): Output tensor of shape (batch_size, seq_length, d_model).
        """
        # Generate Q, K, V using the projection module
        Q, K, V = self.qkv_projection(X)

        # 🔹 TODO: Ensure the following implementation is clear and complete:
        batch_size, num_heads, seq_length, d_k = Q.shape

        # 🔹 TODO: Compute scaled dot-product attention for the raw_scores:
        #    a) Compute the raw attention scores using the dot product between Q and K.
        #    b) Scale the scores by dividing by sqrt(d_k).
        raw_scores = None   # 🔻 REPLACE 'None' with your implementation

        # 🔹 TODO: Apply causal mask if causal_mask is True
        # Hint: Create an upper triangular matrix of ones using torch.triu(..., diagonal=1)
        # and use masked_fill to set future positions to MASK_VALUE BEFORE applying softmax.
        # YOUR CODE HERE

        # 🔹 TODO: Apply the padding attention mask (attention_mask) to the scores
        # The attention_mask has shape (batch_size, seq_length) with 1 for real tokens and 0 for padding.
        # You need to:
        #   1. Identify padding positions (where attention_mask == 0)
        #   2. Expand the mask to shape (batch_size, 1, 1, seq_length) so it broadcasts over heads and query positions
        #   3. Use masked_fill to set those positions to MASK_VALUE BEFORE softmax
        # YOUR CODE HERE

        # 🔹 TODO: Apply softmax to get attention weights
        attention_weights = None   # 🔻 REPLACE 'None' with your implementation

        # Apply dropout to the attention weights only during training
        if self.attn_pdrop is not None and self.training:
            attention_weights = F.dropout(attention_weights, p=self.attn_pdrop, training=self.training)

        # 🔹 TODO: Now we should compute the context vector
        context = None  # 🔻 REPLACE 'None' with your implementation

        # 🔹 TODO: Compute the output vector.
        # Hint: Reshape the context tensor to match the original shape of X.
        output = None  # 🔻 REPLACE 'None' with your implementation

        return attention_weights, output

# Example usage:
d_model = 512  # Embedding size
num_heads = 8  # Number of attention heads
d_k = 64  # Dimension per head (num_heads * d_k must be d_model)

batch_size = 2
seq_length = 10

# Random input tensor
X = torch.rand(batch_size, seq_length, d_model)

# Instantiate and apply Multi-Head Attention
multi_head_attn = MultiHeadAttention(d_model, num_heads, d_k, attn_pdrop=0.0)
_, C = multi_head_attn(X)

print(C.shape)  # Expected: (batch_size, seq_length, d_model)




# ----------- Unit tests ---------------
# -----------DO NOT EDIT THIS PART -----


import unittest
import torch

class TestMultiHeadAttention(unittest.TestCase):
    def setUp(self):
        self.d_model = 512
        self.num_heads = 8
        self.d_k = 64
        self.batch_size = 2
        self.seq_length = 10
        torch.manual_seed(42)
        self.multi_head_attn = MultiHeadAttention(self.d_model, self.num_heads, self.d_k)
        self.X = torch.rand(self.batch_size, self.seq_length, self.d_model, requires_grad=True)

    def test_output_shape(self):
        _, output = self.multi_head_attn(self.X)
        expected_shape = (self.batch_size, self.seq_length, self.d_model)
        self.assertEqual(output.shape, expected_shape, f"Expected shape {expected_shape}, but got {output.shape}")

    def test_deterministic_behavior(self):
        torch.manual_seed(42)
        _, output1 = self.multi_head_attn(self.X)
        torch.manual_seed(42)
        _, output2 = self.multi_head_attn(self.X)
        self.assertTrue(torch.allclose(output1, output2, atol=1e-6), "MultiHeadAttention is not deterministic!")

    def test_gradient_computation(self):
        """Ensure gradients are computed properly."""
        _, output = self.multi_head_attn(self.X)
        loss = output.sum()  # Simple loss function
        loss.backward()

        self.assertIsNotNone(self.X.grad, "Gradients were not computed for input!")
        self.assertGreater(self.X.grad.abs().sum().item(), 0, "Gradient sum is zero!")

    def test_attention_softmax(self):
        """Ensure that the attention scores sum up to ~1."""
        attention_weights, _ = self.multi_head_attn(X)

        # Sum of softmax probabilities along last dimension should be close to 1
        attention_sum = attention_weights.sum(dim=-1)
        ones = torch.ones_like(attention_sum)

        self.assertTrue(torch.allclose(attention_sum, ones, atol=1e-6), "Attention weights do not sum to 1!")

    def test_known_computation(self):
        d_model = 4
        num_heads = 1
        d_k = 4
        seq_length = 3
        batch_size = 1
        multi_head_attn = MultiHeadAttention(d_model, num_heads, d_k)
        with torch.no_grad():
            # For Q, K, V projections
            for linear in [multi_head_attn.qkv_projection.q_linear,
                           multi_head_attn.qkv_projection.k_linear,
                           multi_head_attn.qkv_projection.v_linear,
                           multi_head_attn.out_linear]:
                linear.weight.copy_(torch.eye(d_model))
                if linear.bias is not None:
                    linear.bias.zero_()
        X_known = torch.ones(batch_size, seq_length, d_model)
        attention_weights, output = multi_head_attn(X_known)
        expected_output = torch.ones(batch_size, seq_length, d_model)
        self.assertTrue(torch.allclose(output, expected_output, atol=1e-6),
                        f"Expected output {expected_output}, but got {output}")

    def test_padding_mask(self):
        """Ensure that padding mask prevents attending to padded positions."""
        d_model = 4
        num_heads = 1
        d_k = 4
        seq_length = 4
        batch_size = 1

        multi_head_attn = MultiHeadAttention(d_model, num_heads, d_k)
        X_test = torch.rand(batch_size, seq_length, d_model)
        # Mask out last 2 positions (padding)
        attn_mask = torch.tensor([[1, 1, 0, 0]])
        attention_weights, _ = multi_head_attn(X_test, attention_mask=attn_mask)
        # Attention weights to padded positions should be ~0
        self.assertTrue(
            torch.allclose(attention_weights[:, :, :, 2:], torch.zeros_like(attention_weights[:, :, :, 2:]), atol=1e-6),
            "Attention weights to padded positions should be zero!"
        )

unittest.main(argv=[''], exit=False)

## 2.3 Feedforward network [3 pts]

**Rubric:** Two linear layers with ReLU and dropout.

The feedforward network is a simple two-layer neural network with a ReLU activation function in between the layers.
The main parameters of this layer are the weight matrices $W_1, W_2$ and the bias vectors $b_1, b_2$.
This layer takes as input a sequence of vectors of dimension $d_{model}$ and returns a sequence of vectors of the same dimension.
This layer is applied to each position in the sequence independently.

We'll next implement this module.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super(FeedForward, self).__init__()

        self.dropout = nn.Dropout(dropout)

        # 🔹 TODO Implement the init function. Recall that we need 2 linear layers. We can use nn.Linear


    def forward(self, x: Tensor) -> Tensor:
        # 🔹 TODO: Implement the forward pass
        # YOUR CODE HERE
        return x   # 🔻 REPLACE x with whatever your implementation returns

## 2.4 Layer normalization [3 pts]

**Rubric:** Layer normalization with gamma/beta.

Layer normalization is a simple normalization technique that is applied to the output of each sub-layer in the transformer model.  
Recall that the main hyperparameters of this layer are the scaling and shifting parameters $\gamma, \beta$.  
Next we implement this module.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, hidden_size, eps: float):
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(hidden_size))
        self.beta = nn.Parameter(torch.zeros(hidden_size))
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        # 🔹 TODO: Implement the layer normalization
        # hint: use the formula from the lecture slides and recall the shape of the tensor x: [batch_size, seq_len, d_model]
        # YOUR CODE HERE
        return None  # 🔻 REPLACE 'None' with whatever your implementation returns

## 2.5 Transformer Block [4 pts]

**Rubric:** Combine attention + FF with residual connections and layer norm; pass attention_mask.

Now we can put the multi-head self-attention and the feedforward network together to implement the transformer block.  
Recall that we also need to implement the layer normalization and the residual connections.


You are provided with a code skeleton for the TransformerBlock class. Your task is to implement the forward method to correctly combine the attention mechanism, feed-forward network, residual connections, and layer normalization.
Pay close attention to the following:

Input Parameters:

`d_model`: The dimension of the input embeddings.   
`num_heads`: The number of attention heads.  
`attn_pdrop`: The dropout probability used in the attention module.  
`dropout`: The dropout probability for the feed-forward network.  
`d_ff`: The hidden dimension size in the feed-forward network.  
`eps`: A small constant for numerical stability in layer normalization.  

In [ ]:
# This uses your previous implementation of the previous blocks.
# Remember that we need to use the MultiHeadAttention, FeedForward, and LayerNorm modules that you implemented earlier

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, eps: float):
        super(TransformerBlock, self).__init__()

        # 🔹 TODO: initialize and use any of the modules that are required for a Transformer Block
        # Hint: You need MultiHeadAttention, FeedForward, and LayerNorm modules
        # Hint: For MultiHeadAttention, compute d_k = d_model // num_heads
        # Hint: Remember that MultiHeadAttention returns a tuple (attention_weights, output)
        # YOUR CODE HERE

    def forward(self, x: Tensor, causal_mask: bool, attention_mask=None) -> Tensor:
        """
        Perform the forward pass of the transformer block.

        Args:
            x (Tensor): Input tensor with shape (batch_size, seq_length, d_model).
            causal_mask (bool): If True, apply a causal mask in the self-attention layer
                                (useful for decoder self-attention to prevent attending to future tokens).
            attention_mask (Tensor, optional): Padding mask of shape (batch_size, seq_length).
                Contains 1 for real tokens and 0 for padding tokens. Pass this to MultiHeadAttention.

        Returns:
            Tensor: The output of the transformer block with shape (batch_size, seq_length, d_model).
        """
        # 🔹 TODO Implement the forward pass
        # hint: the forward pass is similar to the one in the lecture slides. You should use the MultiHeadAttention and FeedForward modules that you implemented earlier
        # hint: remember to use residual connections (x + layer_output) after attention and feed-forward
        # hint: remember to apply layer normalization before attention and feed-forward
        # note: transformers originally apply dropout after attention and feed-forward. But modern architectures apply it before
        #       for stability and better performance.
        # hint: remember to apply dropout after attention and feed-forward but before residual
        # hint: pass attention_mask to the MultiHeadAttention forward call
        # YOUR CODE HERE
        return None  # 🔻 REPLACE 'None' with whatever your implementation returns

## 2.6 Transformer Stack [4 pts]

**Rubric:** Stack N TransformerBlocks; pass attention_mask through.

We can now put the transformer blocks on top of each other to implement the transformer stack.  
In this part we will simply stack the transformer blocks on top of each other to implement the transformer stack.

**Key Concepts**
- Sequential processing: Each layer builds upon the representations learned by previous layers
- Parameters of layers: All layers share the same architecture but have different learned parameters
- Depth vs width: The number of layers (depth) is crucial for model capacity, while d_model represents width

In [ ]:
class TransformerStack(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, eps: float):
        """
        A stack of Transformer blocks that processes input sequentially through multiple layers.
        This architecture allows for deep hierarchical processing of input sequences.

        Architecture:
            Input → TransformerBlock₁ → TransformerBlock₂ → ... → TransformerBlockₙ → Output
        """
        super(TransformerStack, self).__init__()

        # 🔹 TODO Implement
        # hint: you should create a list of TransformerBlock modules and store it in self.layers
        self.layers = None  # 🔻 REPLACE 'None' with your implementation


    def forward(self, x: Tensor, causal_mask: bool, attention_mask=None) -> Tensor:
        """
        Process input through all transformer blocks sequentially.

        Implementation steps:
        1. Iterate through each layer in the stack
        2. Pass the output of each layer as input to the next layer
        3. Maintain the causal mask and attention_mask throughout if specified

        Args:
            x (Tensor): Input tensor of shape (batch_size, seq_length, d_model)
            causal_mask (bool): If True, apply causal masking in all attention layers
            attention_mask (Tensor, optional): Padding mask of shape (batch_size, seq_length).
                Pass this through to each TransformerBlock.

        Returns:
            Tensor: Processed output of shape (batch_size, seq_length, d_model)
        """
        # 🔹 TODO: Implement
        # Hint: pass attention_mask to each layer's forward call
        # YOUR CODE HERE
        return x   # 🔻 REPLACE x with whatever your implementation returns

## 2.7 Building a Complete Transformer Model [4 pts]

**Rubric:** Combine embeddings + stack + LM head; pass attention_mask.

Finally we can put the input embeddings and the transformer stack together to implement the transformer model.  
This includes the input pipeline, the transformer stack and the output layer.  

In [ ]:
class TransformerModel(nn.Module):
    """
    Complete Transformer model for language modeling tasks.

    Architecture:
        Input Tokens → Token Embeddings → Transformer Stack → Language Model Head → Output Logits
    """
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, max_len: int, num_layers: int, eps: float):
        super(TransformerModel, self).__init__()

        # 🔹 TODO: Implement
        # hint: you should create the TokenEmbedder and TransformerStack modules
        # hint: at the end, you should add a linear layer to convert the transformer output to the vocabulary size
        # this is the language model head which allows us to predict the next token in the sequence

        # YOUR CODE HERE

    def forward(self, x: Tensor, causal_mask: bool, attention_mask=None) -> Tensor:
        """
        Forward pass of the transformer model.

        Args:
            x (Tensor): Input token IDs of shape (batch_size, seq_length).
            causal_mask (bool): If True, apply causal masking in attention layers.
            attention_mask (Tensor, optional): Padding mask of shape (batch_size, seq_length).
                Contains 1 for real tokens and 0 for padding tokens. Pass this to TransformerStack.

        Returns:
            Tensor: Logits of shape (batch_size, seq_length, vocab_size).
        """
        # 🔹 TODO: Implement
        # Hint: pass attention_mask through to the transformer_stack
        # YOUR CODE HERE
        return None   # 🔻 REPLACE None with whatever your implementation returns

Let's test your model!

In [ ]:
# hyperparameters

vocab_size = tokenizer.vocab_size
d_model = 768
num_heads = 12
attn_pdrop = 0.1
dropout = 0.1
d_ff = 3072
max_len = 20
num_layers = 12
eps = 1e-6

# create the model
model = TransformerModel(vocab_size, d_model, num_heads, attn_pdrop, dropout, d_ff, max_len, num_layers, eps)

encoded = tokenizer(sample_texts, return_tensors="pt", max_length=max_len, padding="longest", truncation=True)
token_ids = encoded['input_ids']
attention_mask = encoded['attention_mask']
output = model(token_ids, causal_mask=True, attention_mask=attention_mask)

## 3. Reflection questions [4 pts, first 2 questions 1 point each, last question 2 points]

Please answer the following questions in the markdown cells below.

1- What is the purpose of the output projection $W_O$ in the transformer model?

TODO: Your answer here

2 - What is the total number of trainable parameters in the multi-head self-attention layer that you implemneted above?  
Provide the final answer first, and then show your work.

TODO: Your answer here

3- What is the time complexity of the multi-head self-attention layer that you implemented above?  
Provide your answer in terms of the sequence length $n$, the dimension of the input sequence $d_{model}$.
Also show your work on how you arrived at your answer.

TODO: Your answer here

In [ ]:
raise NotImplementedError, "First answer the above questions and then comment this line to continue"

## 4. Training Your Transformer [12 pts]

**Rubric:** Implement training with a given DataLoader and implement a simple training loop with learning rate scheduler.

Now that you have implemented the full transformer architecture, let's verify it actually learns!
We'll create a small transformer and train it on a language modeling task for 100 steps.
If your implementation is correct, you should see the training loss decrease steadily.

We'll use the [WikiText-2](https://huggingface.co/datasets/Salesforce/wikitext) dataset, a standard small benchmark for language modeling.

In [ ]:
!pip install --upgrade datasets huggingface_hub fsspec
from datasets import load_dataset

# Load a small language modeling dataset
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# Filter out empty lines and very short texts
train_texts = [x['text'] for x in wiki_dataset if len(x['text'].strip()) > 50]
print(f"Number of training texts: {len(train_texts)}")
print(f"Example: {train_texts[0][:200]}")

In [ ]:
# Data loading code
# DO NOT EDIT THIS CELL

from torch.utils.data import Dataset, DataLoader

tokenizer.pad_token = tokenizer.eos_token

class TextDataset(Dataset):
    """Simple dataset that wraps a list of text strings."""
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

seq_len = 64
batch_size = 8

def collate_fn(batch_texts):
    # The collate function prepares each batch for model input by tokenizing and padding all texts
    # in the batch to a fixed maximum length. It returns tensors of input token ids and attention masks,
    # ensuring consistent batch shapes and properly masked positions for the transformer.
    """Tokenize and pad a batch of texts."""
    encoded = tokenizer(batch_texts, return_tensors='pt', max_length=seq_len,
                        padding='max_length', truncation=True)
    return encoded['input_ids'], encoded['attention_mask']

train_dataset = TextDataset(train_texts)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
print(f"Dataset size: {len(train_dataset)}, Number of batches per epoch: {len(train_loader)}")

### Small Model Configuration

We'll create a **small** transformer to keep training fast. Instead of GPT-2's 12 layers with d_model=768,
we'll use just 4 layers with d_model=128. This makes training feasible in a few minutes even on CPU.

In [ ]:
# Small model hyperparameters (4 layers, d_model=128 for fast training)
small_config = {
    "vocab_size": tokenizer.vocab_size,
    "d_model": 128,
    "num_heads": 4,
    "attn_pdrop": 0.1,
    "dropout": 0.1,
    "d_ff": 512,
    "max_len": seq_len,
    "num_layers": 4,
    "eps": 1e-6,
}
small_model = TransformerModel(**small_config).to(DEVICE)

### 4.1 Training Loop [6 pts]

Implement a simple training loop for language modeling using the `DataLoader`. At each step:
1. Get the next batch from the `DataLoader` (handle `StopIteration` by resetting the iterator)
2. Move tensors to `DEVICE`
3. Create input/target pairs: input is `tokens[:, :-1]`, target is `tokens[:, 1:]` (shifted by one position). Also shift the `attention_mask` the same way (drop last position).
4. Forward pass through the model **with causal masking enabled** (think about why) and with `attention_mask`
5. Compute cross-entropy loss between model predictions and target tokens
   You need to ignore computing loss on pad tokens. For this you can set `ignore_index` to the pad token to skip padding
6. Backward pass and optimizer step
   For backward pass you need the following steps:
   First set the gradients to zero: `optimizer.zero_grad()`
   Then compute the loss and call backward: `loss.backward()`
   Finally call the optimizer step to update the model parameters: `optimizer.step()`

Use `Adam` optimizer with learning rate `1e-3` and train for `100` steps.
Log the loss every 10 steps and plot the loss curve at the end.

**If your implementation is correct**, you should see:
- Initial loss around **10.8** (which is $\log(50257)$, the log of the vocabulary size)
- Loss steadily decreasing over the 100 steps although some fluctuations are expected

In [ ]:
# STARTER CODE

import matplotlib.pyplot as plt

small_model.train()
optimizer = torch.optim.Adam(small_model.parameters(), lr=1e-3)

num_steps = 100
vocab_size = tokenizer.vocab_size

losses = []
train_iter = iter(train_loader)

for step in range(num_steps):
    # TODO: Implement the training loop
    # 1. Get the next batch from the DataLoader:
    #    Move tokens and attention_mask to DEVICE
    # 2. Split the batch into input (all tokens in the sequence but the last token) and target (all tokens in the sequence but the first token)
    #    Also shift the attention_mask the same way as input_ids (drop last position)
    # 3. Forward pass with causal_mask=True and attention_mask=input_attention_mask
    # 4. Compute cross-entropy loss:
    #    F.cross_entropy(...)
    #    Note: `ignore_index` look up the documentation of F.cross_entropy
    # 5. Call the backward pass (see notes above)
    # 6. Append loss.item() to losses list (for logging)

    # 🔹 TODO Implement
    # YOUR CODE HERE

    if (step + 1) % 10 == 0:
        print(f"Step {step+1}/{num_steps}, Loss: {losses[-1]:.4f}")

# Plot the training loss
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nInitial loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Expected initial loss (log(vocab_size)): {math.log(tokenizer.vocab_size):.4f}")

### 4.1 Reflection Questions [3 pts, 1pt each]

1- Describe the training loss curve. Does the model appear to be learning? What does the initial loss value correspond to and why?

`TODO: Your answer here`

2- How many parameters does your small 4-layer model have? How does this compare to GPT-2 (117M parameters)? What would you need to change to scale up your model to GPT-2 size?

`TODO: Your answer here`

3- Why is the causal mask essential for training a language model? What would happen if you removed it?

`TODO: Your answer here`

In [ ]:
raise NotImplementedError, "First answer the above questions and then comment this line to continue"

### 4.2 Learning Rate Scheduler [3 pts]

**Rubric:** Implement `get_lr_multiplier(step, warmup_steps, total_steps)` with linear warmup + linear decay. In the next cell, create a `LambdaLR` scheduler using this function and call `scheduler.step()` in the training loop.

A common practice in transformer training is to use a **learning rate schedule with warmup**:
- **Warmup** (steps `0` to `warmup_steps`): Linearly increase the learning rate from 0 to the base LR.
- **Decay** (steps `warmup_steps` to `total_steps`): Linearly decrease the learning rate from the base LR down to 0.

**Your task:** Implement a function that returns a **multiplier** between 0 and 1 for the current step. The actual learning rate will be `base_lr * multiplier`. PyTorch's `LambdaLR` will call your function with `step = 0, 1, 2, ...` after each optimizer step.

**Target values (for sanity checks):**
| Step | Multiplier |
|------|------------|
| 0 | 0 |
| warmup_steps | 1 |
| total_steps - 1 | ~0 (linear decay) |

**Formulas:**
- **Warmup** (`step < warmup_steps`): multiplier increases linearly from 0 to 1 as step goes from 0 to warmup_steps.
- **Decay** (`step >= warmup_steps`): multiplier decreases linearly from 1 to 0 as step goes from warmup_steps to total_steps (use the number of steps in the decay phase in the denominator).

In [ ]:
def get_lr_multiplier(step, warmup_steps, total_steps):
    """
    Return the learning rate multiplier for the given step (used as lr = base_lr * multiplier).

    - Warmup (step < warmup_steps): multiplier goes from 0 to 1.
    - Decay (step >= warmup_steps): multiplier goes from 1 to 0.

    Args:
        step (int): Current training step (0-indexed).
        warmup_steps (int): Number of warmup steps.
        total_steps (int): Total number of training steps.

    Returns:
        float: Multiplier in [0, 1].
    """
    if step < warmup_steps:
        # 🔹 TODO: Linear warmup — multiplier from 0 at step 0 to 1 at step warmup_steps
        # Hint: multiplier = step / warmup_steps (handle warmup_steps == 0 if needed)
        return None  # 🔻 REPLACE with your formula
    else:
        # 🔹 TODO: Linear decay — multiplier from 1 at step warmup_steps to 0 at step total_steps
        # Hint: decay_steps = total_steps - warmup_steps; then linear decrease from 1 to 0
        # Hint: use max(0.0, ...) so the multiplier never goes negative
        return None  # 🔻 REPLACE with your formula

In [ ]:
# Retrain with LR scheduler for longer
small_model_sched = TransformerModel(**small_config).to(DEVICE)
small_model_sched.train()

base_lr = 5e-4
num_steps_sched = 500
warmup_steps = 50

optimizer_sched = torch.optim.Adam(small_model_sched.parameters(), lr=base_lr)

# 🔹 TODO: Create a LambdaLR scheduler using your get_lr_multiplier function
# 💡 Hint: Use torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=...)
# 💡 Hint: The lr_lambda should be a function that takes a step and returns the multiplier
scheduler = None  # 🔻 REPLACE with your implementation

losses_sched = []
lr_history = []
train_iter = iter(train_loader)

for step in range(num_steps_sched):
    try:
        tokens, attention_mask = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        tokens, attention_mask = next(train_iter)
    tokens = tokens.to(DEVICE)
    attention_mask = attention_mask.to(DEVICE)

    input_ids = tokens[:, :-1]
    target_ids = tokens[:, 1:]
    input_attention_mask = attention_mask[:, :-1]

    logits = small_model_sched(input_ids, causal_mask=True, attention_mask=input_attention_mask)
    loss = F.cross_entropy(
        logits.view(-1, tokenizer.vocab_size),
        target_ids.reshape(-1),
        ignore_index=tokenizer.pad_token_id
    )

    optimizer_sched.zero_grad()
    loss.backward()
    optimizer_sched.step()

    # 🔹 TODO: Step the learning rate scheduler
    # YOUR CODE HERE

    losses_sched.append(loss.item())
    lr_history.append(optimizer_sched.param_groups[0]['lr'])

    if (step + 1) % 50 == 0:
        print(f"Step {step+1}/{num_steps_sched}, Loss: {loss.item():.4f}, LR: {lr_history[-1]:.6f}")

# Plot the training loss and learning rate
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(losses_sched)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (with LR Scheduler)')
ax1.grid(True, alpha=0.3)

ax2.plot(lr_history)
ax2.axvline(x=warmup_steps, color='r', linestyle='--', label=f'End of warmup (step {warmup_steps})')
ax2.set_xlabel('Step')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInitial loss: {losses_sched[0]:.4f}")
print(f"Final loss: {losses_sched[-1]:.4f}")

In [ ]:
# Visualize the learning rate schedule
import matplotlib.pyplot as plt

warmup_steps = 10
total_steps = 100
base_lr = 1e-3

steps = list(range(total_steps))
lr_values = [base_lr * get_lr_multiplier(s, warmup_steps, total_steps) for s in steps]

plt.figure(figsize=(10, 4))
plt.plot(steps, lr_values, linewidth=2)
plt.axvline(x=warmup_steps, color='r', linestyle='--', label=f'End of warmup (step {warmup_steps})')
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule: Linear Warmup + Linear Decay')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()